In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("MySQL_to_HDFS")
    .enableHiveSupport()
    .getOrCreate()
)

spark

#### Transfer Data to Big Data Environment using spark JDBC

In [3]:
jdbc_url = "jdbc:mysql://mysql:3306/source_db"

properties = {
    "user": "appuser",
    "password": "apppass",
    "driver": "com.mysql.cj.jdbc.Driver"
}

In [13]:
employees_df = spark.read.jdbc(
    url=jdbc_url,
    table="employees",
    properties=properties
)

cities_df = spark.read.jdbc(
    url=jdbc_url,
    table="cities",
    properties=properties
)
departments_df = spark.read.jdbc(
    url=jdbc_url,
    table="departments",
    properties=properties
)

In [9]:
employees_df.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- city_id: integer (nullable = true)
 |-- job_title: string (nullable = true)
 |-- salary: decimal(10,2) (nullable = true)
 |-- bonus: decimal(10,2) (nullable = true)
 |-- performance_score: decimal(3,2) (nullable = true)
 |-- employment_status: string (nullable = true)



In [14]:
cities_df.printSchema()

root
 |-- city_id: integer (nullable = true)
 |-- city_name: string (nullable = true)



In [15]:
departments_df.printSchema()

root
 |-- department_id: integer (nullable = true)
 |-- department_name: string (nullable = true)



In [10]:
employees_df.show(5)

+-----------+----------+---------+------+----------+----------+-------------+-------+--------------------+--------+--------+-----------------+-----------------+
|employee_id|first_name|last_name|gender|birth_date| hire_date|department_id|city_id|           job_title|  salary|   bonus|performance_score|employment_status|
+-----------+----------+---------+------+----------+----------+-------------+-------+--------------------+--------+--------+-----------------+-----------------+
|          1|    Rhonda| Thompson|     F|1982-12-07|2018-12-13|            2|      2|          HR Manager|15139.00| 5964.77|             3.94|           Active|
|          2|   Vincent|     Hall|     M|1969-12-25|2025-07-15|            9|      3|    Support Engineer| 9965.00| 4075.68|             4.09|           Active|
|          3|    Amanda|    Burch|     F|1967-04-18|2025-02-12|            1|     11|   Software Engineer|23642.00| 9835.07|             4.16|           Active|
|          4|   Tiffany|    Blake|

In [11]:
employees_df.count()

120000

In [16]:
employees_df.write.mode("overwrite").parquet("hdfs://namenode:8020/data/raw/employees")

In [17]:
cities_df.write.mode("overwrite").parquet("hdfs://namenode:8020/data/raw/cities")

In [18]:
departments_df.write.mode("overwrite").parquet("hdfs://namenode:8020/data/raw/departments")

### Process Data with Spark

In [23]:
employees_raw = spark.read.parquet("hdfs://namenode:8020/data/raw/employees")

In [29]:
from pyspark.sql.functions import (
    year,
    current_date,
    months_between,
    round,
    when,
    col,
    floor
)

employees_processed = (
    employees_raw

    #extract year from the date
    .withColumn("hire_year", year("hire_date"))

    # Employee age
    .withColumn(
        "age",
        floor(months_between(current_date(), col("birth_date")) / 12)
    )

    # Salary category
    .withColumn(
        "salary_band",
        when(col("salary") < 10000, "Low")
        .when(col("salary") < 20000, "Medium")
        .otherwise("High")
    )
)


In [30]:
employees_processed.show(5)

+-----------+----------+---------+------+----------+----------+-------------+-------+--------------------+--------+--------+-----------------+-----------------+---------+---+-----------+
|employee_id|first_name|last_name|gender|birth_date| hire_date|department_id|city_id|           job_title|  salary|   bonus|performance_score|employment_status|hire_year|age|salary_band|
+-----------+----------+---------+------+----------+----------+-------------+-------+--------------------+--------+--------+-----------------+-----------------+---------+---+-----------+
|          1|    Rhonda| Thompson|     F|1982-12-07|2018-12-13|            2|      2|          HR Manager|15139.00| 5964.77|             3.94|           Active|     2018| 43|     Medium|
|          2|   Vincent|     Hall|     M|1969-12-25|2025-07-15|            9|      3|    Support Engineer| 9965.00| 4075.68|             4.09|           Active|     2025| 56|        Low|
|          3|    Amanda|    Burch|     F|1967-04-18|2025-02-12|  

In [31]:
#write to /processed
employees_processed.write \
    .mode("overwrite") \
    .partitionBy("hire_year") \
    .parquet("hdfs://namenode:8020/data/processed/employees")


In [26]:
#for cities
spark.read.parquet("hdfs://namenode:8020/data/raw/cities") \
    .write.mode("overwrite") \
    .parquet("hdfs://namenode:8020/data/processed/cities")


In [27]:
#for departments
spark.read.parquet("hdfs://namenode:8020/data/raw/departments") \
    .write.mode("overwrite") \
    .parquet("hdfs://namenode:8020/data/processed/departments")

### build warehouse

In [33]:
employees = spark.read.parquet("hdfs://namenode:8020/data/processed/employees")
departments = spark.read.parquet("hdfs://namenode:8020/data/processed/departments")
cities = spark.read.parquet("hdfs://namenode:8020/data/processed/cities")

In [38]:
#Create Employee Dimension
dim_employee = employees.select(
    "employee_id",
    "first_name",
    "last_name",
    "gender",
    "birth_date",
    "job_title",
    "employment_status",
    "salary_band"
)

In [39]:
dim_employee.show(5)

+-----------+----------+---------+------+----------+--------------------+-----------------+-----------+
|employee_id|first_name|last_name|gender|birth_date|           job_title|employment_status|salary_band|
+-----------+----------+---------+------+----------+--------------------+-----------------+-----------+
|          2|   Vincent|     Hall|     M|1969-12-25|    Support Engineer|           Active|        Low|
|          3|    Amanda|    Burch|     F|1967-04-18|   Software Engineer|           Active|       High|
|          5|    Joseph|  Salinas|     M|2002-12-06|       Sales Manager|           Active|     Medium|
|         12|     Megan|  Carroll|     F|1995-04-06|      Cloud Engineer|         Resigned|       High|
|         23|  Samantha|   Cortez|     F|1984-10-24|Senior Software E...|           Active|       High|
+-----------+----------+---------+------+----------+--------------------+-----------------+-----------+
only showing top 5 rows



In [40]:
#Create Department Dimension
dim_department = departments.select(
    "department_id",
    "department_name"
)

In [41]:
#Create City Dimension
dim_city = cities.select(
    "city_id",
    "city_name"
)

In [43]:
#Create Fact Table
fact_employee = employees.select(
    "employee_id",
    "department_id",
    "city_id",
    "hire_year",
    "salary",
    "bonus",
    "performance_score"
)

In [44]:
fact_employee.show(5)

+-----------+-------------+-------+---------+--------+--------+-----------------+
|employee_id|department_id|city_id|hire_year|  salary|   bonus|performance_score|
+-----------+-------------+-------+---------+--------+--------+-----------------+
|          2|            9|      3|     2025| 9965.00| 4075.68|             4.09|
|          3|            1|     11|     2025|23642.00| 9835.07|             4.16|
|          5|            6|      1|     2025|18098.00| 6605.77|             3.65|
|         12|            5|      2|     2025|23521.00| 9173.19|             3.90|
|         23|            1|      1|     2025|28521.00|11493.96|             4.03|
+-----------+-------------+-------+---------+--------+--------+-----------------+
only showing top 5 rows



In [47]:
#Write Employee Dimension
dim_employee.write \
    .mode("overwrite") \
    .parquet("hdfs://namenode:8020/data/warehouse/dim_employee")

In [48]:
# Department Dimension
dim_department.write \
    .mode("overwrite") \
    .parquet("hdfs://namenode:8020/data/warehouse/dim_department")

In [49]:
# City Dimension
dim_city.write \
    .mode("overwrite") \
    .parquet("hdfs://namenode:8020/data/warehouse/dim_city")

In [60]:
# Fact Table
fact_employee.write \
    .mode("overwrite") \
    .partitionBy("hire_year")\
    .parquet("hdfs://namenode:8020/data/warehouse/fact_employee")

### Hive tables

In [52]:
# Create database
spark.sql("""
CREATE DATABASE IF NOT EXISTS hr_dw
""")

DataFrame[]

In [53]:
spark.sql("""
USE hr_dw
""")

DataFrame[]

In [85]:
spark.sql("""
DROP TABLE IF EXISTS hr_dw.fact_employee
""")

DataFrame[]

In [86]:
spark.sql("""
CREATE EXTERNAL TABLE hr_dw.fact_employee (

    employee_id INT,
    department_id INT,
    city_id INT,

    salary DECIMAL(10,2),
    bonus DECIMAL(10,2),
    performance_score DECIMAL(3,2)

)
PARTITIONED BY (
    hire_year INT
)
STORED AS PARQUET
LOCATION '/data/warehouse/fact_employee'
""")


DataFrame[]

In [87]:
spark.sql("""
MSCK REPAIR TABLE fact_employee
""")

DataFrame[]

In [88]:
spark.sql("""
SHOW PARTITIONS hr_dw.fact_employee
""").show(truncate=False)

+--------------+
|partition     |
+--------------+
|hire_year=2015|
|hire_year=2016|
|hire_year=2017|
|hire_year=2018|
|hire_year=2019|
|hire_year=2020|
|hire_year=2021|
|hire_year=2022|
|hire_year=2023|
|hire_year=2024|
|hire_year=2025|
+--------------+



In [83]:
spark.sql("""
SELECT COUNT(*)
FROM hr_dw.fact_employee
""").show()

+--------+
|count(1)|
+--------+
|  120000|
+--------+



In [65]:
spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS dim_department (

    department_id INT,
    department_name STRING

)
STORED AS PARQUET
LOCATION '/data/warehouse/dim_department'
""")

DataFrame[]

In [66]:
spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS dim_city (

    city_id INT,
    city_name STRING

)
STORED AS PARQUET
LOCATION '/data/warehouse/dim_city'
""")

DataFrame[]

In [67]:
spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS dim_employee (

    employee_id INT,
    first_name STRING,
    last_name STRING,
    gender STRING,
    birth_date DATE,
    job_title STRING,
    employment_status STRING,
    salary_band STRING

)
STORED AS PARQUET
LOCATION '/data/warehouse/dim_employee'
""")

DataFrame[]

In [70]:
spark.read.parquet(
    "hdfs://namenode:8020/data/warehouse/fact_employee"
).printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- city_id: integer (nullable = true)
 |-- salary: decimal(10,2) (nullable = true)
 |-- bonus: decimal(10,2) (nullable = true)
 |-- performance_score: decimal(3,2) (nullable = true)
 |-- hire_year: integer (nullable = true)



In [69]:
spark.sql("""
DESCRIBE hr_dw.fact_employee
""").show(truncate=False)

+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|employee_id            |int      |NULL   |
|department_id          |int      |NULL   |
|city_id                |int      |NULL   |
|salary                 |int      |NULL   |
|bonus                  |double   |NULL   |
|performance_score      |double   |NULL   |
|hire_year              |int      |NULL   |
|# Partition Information|         |       |
|# col_name             |data_type|comment|
|hire_year              |int      |NULL   |
+-----------------------+---------+-------+



In [78]:
spark.sql("""
DESCRIBE hr_dw.fact_employee
""").show(truncate=False)

+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|employee_id            |int      |NULL   |
|department_id          |int      |NULL   |
|city_id                |int      |NULL   |
|salary                 |bigint   |NULL   |
|bonus                  |double   |NULL   |
|performance_score      |double   |NULL   |
|hire_year              |int      |NULL   |
|# Partition Information|         |       |
|# col_name             |data_type|comment|
|hire_year              |int      |NULL   |
+-----------------------+---------+-------+



In [89]:
spark.sql("SHOW TABLES IN hr_dw").show(truncate=False)


+---------+--------------+-----------+
|namespace|tableName     |isTemporary|
+---------+--------------+-----------+
|hr_dw    |dim_city      |false      |
|hr_dw    |dim_department|false      |
|hr_dw    |dim_employee  |false      |
|hr_dw    |fact_employee |false      |
+---------+--------------+-----------+

+--------+
|count(1)|
+--------+
|  120000|
+--------+

+--------+
|count(1)|
+--------+
|  120000|
+--------+

+-----------+-------------+-------+--------+--------+-----------------+---------+
|employee_id|department_id|city_id|  salary|   bonus|performance_score|hire_year|
+-----------+-------------+-------+--------+--------+-----------------+---------+
|          2|            9|      3| 9965.00| 4075.68|             4.09|     2025|
|          3|            1|     11|23642.00| 9835.07|             4.16|     2025|
|          5|            6|      1|18098.00| 6605.77|             3.65|     2025|
|         12|            5|      2|23521.00| 9173.19|             3.90|     2025|
|

### Run SQL Query Against Hive Table

In [90]:
spark.sql("""
SELECT
    department_id,
    COUNT(*) AS total_employees,
    ROUND(AVG(salary),2) AS avg_salary
FROM hr_dw.fact_employee
GROUP BY department_id
ORDER BY total_employees DESC
""").show()

+-------------+---------------+----------+
|department_id|total_employees|avg_salary|
+-------------+---------------+----------+
|            1|          35925|  28670.55|
|            6|          17977|  17252.68|
|            5|          14327|  22494.51|
|            3|          12279|  18980.61|
|            4|           9581|  16236.06|
|            7|           9575|  17270.04|
|            9|           8412|  11002.04|
|            2|           7134|  11748.49|
|           10|           2431|  30883.85|
|            8|           2359|  24950.47|
+-------------+---------------+----------+



In [91]:
#Query Using the Partition Columns
park.sql("""
SELECT
    COUNT(*)
FROM hr_dw.fact_employee
WHERE hire_year = 2023
""").show()

+--------+
|count(1)|
+--------+
|   12085|
+--------+



In [92]:
#Partition Pruning
spark.sql("""
EXPLAIN
SELECT *
FROM hr_dw.fact_employee
WHERE hire_year = 2023
""").show(truncate=False)


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [93]:
print(
    spark.sql("""
    EXPLAIN
    SELECT *
    FROM hr_dw.fact_employee
    WHERE hire_year = 2023
    """).collect()[0][0]
)

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet spark_catalog.hr_dw.fact_employee[employee_id#1833,department_id#1834,city_id#1835,salary#1836,bonus#1837,performance_score#1838,hire_year#1839] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:8020/data/warehouse/fact_employee/hire_year=2023], PartitionFilters: [isnotnull(hire_year#1839), (hire_year#1839 = 2023)], PushedFilters: [], ReadSchema: struct<employee_id:int,department_id:int,city_id:int,salary:decimal(10,2),bonus:decimal(10,2),per...




In [95]:
spark.sql("""
EXPLAIN
SELECT *
FROM hr_dw.fact_employee
WHERE salary > 25000
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                                                                                                    

In [94]:
#Compare With a Non-Partition Query
print(
    spark.sql("""
EXPLAIN
SELECT *
FROM hr_dw.fact_employee
WHERE salary > 25000
""").collect()[0][0]
     )

== Physical Plan ==
*(1) Filter (isnotnull(salary#1848) AND (salary#1848 > 25000.00))
+- *(1) ColumnarToRow
   +- FileScan parquet spark_catalog.hr_dw.fact_employee[employee_id#1845,department_id#1846,city_id#1847,salary#1848,bonus#1849,performance_score#1850,hire_year#1851] Batched: true, DataFilters: [isnotnull(salary#1848), (salary#1848 > 25000.00)], Format: Parquet, Location: CatalogFileIndex(1 paths)[hdfs://namenode:8020/data/warehouse/fact_employee], PartitionFilters: [], PushedFilters: [IsNotNull(salary), GreaterThan(salary,25000.00)], ReadSchema: struct<employee_id:int,department_id:int,city_id:int,salary:decimal(10,2),bonus:decimal(10,2),per...




In [96]:
spark.conf.get("spark.sql.extensions")

'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions'

In [97]:
spark.conf.get("spark.sql.catalog.spark_catalog")

'org.apache.iceberg.spark.SparkSessionCatalog'

In [98]:
print(spark.conf.get("spark.sql.catalogImplementation"))

hive


In [99]:
spark.sql("SHOW DATABASES").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
|hr_dw    |
+---------+



In [100]:
#Create Iceberg Database
spark.sql("""
CREATE DATABASE IF NOT EXISTS hr_iceberg
""")

DataFrame[]

In [101]:
spark.sql("SHOW DATABASES").show(truncate=False)

+----------+
|namespace |
+----------+
|default   |
|hr_dw     |
|hr_iceberg|
+----------+



In [103]:
fact_employee.printSchema()

fact_employee.show(5)

fact_employee.count()

root
 |-- employee_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- city_id: integer (nullable = true)
 |-- salary: decimal(10,2) (nullable = true)
 |-- bonus: decimal(10,2) (nullable = true)
 |-- performance_score: decimal(3,2) (nullable = true)
 |-- hire_year: integer (nullable = true)

+-----------+-------------+-------+--------+--------+-----------------+---------+
|employee_id|department_id|city_id|  salary|   bonus|performance_score|hire_year|
+-----------+-------------+-------+--------+--------+-----------------+---------+
|          2|            9|      3| 9965.00| 4075.68|             4.09|     2025|
|          3|            1|     11|23642.00| 9835.07|             4.16|     2025|
|          5|            6|      1|18098.00| 6605.77|             3.65|     2025|
|         12|            5|      2|23521.00| 9173.19|             3.90|     2025|
|         23|            1|      1|28521.00|11493.96|             4.03|     2025|
+-----------+---------

120000

In [104]:
#create iceberg table
fact_employee.writeTo(
    "hr_iceberg.fact_employee"
) \
.partitionedBy("hire_year") \
.using("iceberg") \
.createOrReplace()

In [105]:
spark.sql("""
SHOW TABLES IN hr_iceberg
""").show(truncate=False)

+----------+-------------+-----------+
|namespace |tableName    |isTemporary|
+----------+-------------+-----------+
|hr_iceberg|fact_employee|false      |
+----------+-------------+-----------+



In [106]:
#Query Iceberg
spark.sql("""
SELECT
    department_id,
    COUNT(*) AS total_employees,
    ROUND(AVG(salary),2) AS avg_salary
FROM hr_iceberg.fact_employee
GROUP BY department_id
ORDER BY total_employees DESC
""").show()

+-------------+---------------+----------+
|department_id|total_employees|avg_salary|
+-------------+---------------+----------+
|            1|          35925|  28670.55|
|            6|          17977|  17252.68|
|            5|          14327|  22494.51|
|            3|          12279|  18980.61|
|            4|           9581|  16236.06|
|            7|           9575|  17270.04|
|            9|           8412|  11002.04|
|            2|           7134|  11748.49|
|           10|           2431|  30883.85|
|            8|           2359|  24950.47|
+-------------+---------------+----------+



In [107]:
spark.sql("""
SELECT COUNT(*)
FROM hr_iceberg.fact_employee
WHERE hire_year = 2023
""").show()

+--------+
|count(1)|
+--------+
|   12085|
+--------+



In [108]:
spark.sql("""
EXPLAIN
SELECT *
FROM hr_iceberg.fact_employee
WHERE hire_year = 2023
""").show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                                                                                                                       |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [109]:
#query time in hive
import time

start = time.time()

spark.sql("""
SELECT COUNT(*)
FROM hr_dw.fact_employee
WHERE hire_year = 2023
""").show()

print(f"Hive Query Time: {time.time()-start:.3f} sec")

+--------+
|count(1)|
+--------+
|   12085|
+--------+

Hive Query Time: 0.427 sec


In [110]:
#query time in iceberg
start = time.time()

spark.sql("""
SELECT COUNT(*)
FROM hr_iceberg.fact_employee
WHERE hire_year = 2023
""").show()

print(f"Iceberg Query Time: {time.time()-start:.3f} sec")


+--------+
|count(1)|
+--------+
|   12085|
+--------+

Iceberg Query Time: 0.261 sec
